# TASK 1

In [1]:
import numpy as np

In [15]:
sites = ["GAGGTAAAC", "TCCGTAAGC", "CAGGTTGGA", "ACAGTCAGC", "TAGGTCAGC", "CAGGTCAGC", "CAGGTCGAT", "CAGGTCAGC", "CAGGTCAGC", "CAGGTTGGC"]

bases = ['A', 'C', 'G', 'T']
base_to_idx = {base: i for i, base in enumerate(bases)}

def create_pfm(sequences, bases):
    n_seqs = len(sequences)
    seq_len = len(sequences[0])
    n_bases = len(bases)
    
    base_to_idx = {base: i for i, base in enumerate(bases)}

    pfm = np.zeros((n_bases, seq_len), dtype=int)

    for seq in sequences:
        for pos, base in enumerate(seq):
            if base in base_to_idx:
                pfm[base_to_idx[base], pos] += 1
    
    return pfm

In [5]:
pfm = create_pfm(sites, bases)
print(pfm)

[[ 1  8  1  0  0  2  7  2  1]
 [ 6  2  1  0  0  6  0  0  8]
 [ 1  0  8 10  0  0  3  8  0]
 [ 2  0  0  0 10  2  0  0  1]]


In [6]:
def pfm_to_ppm(pfm, alpha=0.1):
    n_seqs = np.sum(pfm[:, 0])  # количество последовательностей
    n_bases = pfm.shape[0]
    # добавляем псевдосчёт и нормализуем
    ppm = (pfm + alpha) / (n_seqs + alpha * n_bases)
    return ppm

In [7]:
ppm = pfm_to_ppm(pfm, alpha=0.1)
print(ppm)

[[0.10576923 0.77884615 0.10576923 0.00961538 0.00961538 0.20192308
  0.68269231 0.20192308 0.10576923]
 [0.58653846 0.20192308 0.10576923 0.00961538 0.00961538 0.58653846
  0.00961538 0.00961538 0.77884615]
 [0.10576923 0.00961538 0.77884615 0.97115385 0.00961538 0.00961538
  0.29807692 0.77884615 0.00961538]
 [0.20192308 0.00961538 0.00961538 0.00961538 0.97115385 0.20192308
  0.00961538 0.00961538 0.10576923]]


In [8]:
def ppm_to_pwm(ppm, background_probs):
    n_bases, seq_len = ppm.shape
    # преобразуем background_probs в массив, если это словарь
    if isinstance(background_probs, dict):
        bg_array = np.array([background_probs[base] for base in bases])
    else:
        bg_array = np.array(background_probs)
    
    # Расчет PWM: log2(ppm / background)
    epsilon = 1e-10
    pwm = np.log2((ppm + epsilon) / (bg_array[:, np.newaxis] + epsilon))
    
    return pwm

In [11]:
background = {
    'A': 0.295,
    'C': 0.205,
    'G': 0.205,
    'T': 0.295
}
pwm = ppm_to_pwm(ppm, background)
print(pwm)

[[-1.47979496  1.40062342 -1.47979496 -4.93922656 -4.93922656 -0.54690915
   1.21052054 -0.54690915 -1.47979496]
 [ 1.5166018  -0.02181811 -0.95470391 -4.41413552 -4.41413552  1.5166018
  -4.41413552 -4.41413552  1.92571447]
 [-0.95470391 -4.41413552  1.92571447  2.24407595 -4.41413552 -4.41413552
   0.54006078  1.92571447 -4.41413552]
 [-0.54690915 -4.93922656 -4.93922656 -4.93922656  1.7189849  -0.54690915
  -4.93922656 -4.93922656 -1.47979496]]


In [12]:
def find_extreme_scores(pwm, alphabet):
    seq_len = pwm.shape[1]
    
    # Для максимального скора выбираем на каждой позиции максимальное значение
    max_indices = np.argmax(pwm, axis=0)
    max_scores = np.max(pwm, axis=0)
    max_score = np.sum(max_scores)
    max_sequence = ''.join([alphabet[idx] for idx in max_indices])
    
    # Для минимального скора выбираем на каждой позиции минимальное значение
    min_indices = np.argmin(pwm, axis=0)
    min_scores = np.min(pwm, axis=0)
    min_score = np.sum(min_scores)
    min_sequence = ''.join([alphabet[idx] for idx in min_indices])
    
    return max_score, max_sequence, min_score, min_sequence

In [14]:
print(find_extreme_scores(pwm, bases))

(np.float64(15.384551836632848), 'CAGGTCAGC', np.float64(-39.94342537486369), 'ATTAAGTTG')


# Task 2

In [22]:
from Bio import SeqIO
from Bio import motifs
from Bio.Seq import Seq

In [21]:
data = list(SeqIO.parse('/Users/veronikaaksinina/Documents/bioinf_sem_26/chr21.fasta', 'fasta'))